# Empirical Risk, Generalization Gap, and Validation Splits Lab


## Purpose

This lab compares training, validation, test, and shifted-deployment performance.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 4000

evaluation = pd.DataFrame({
    "example_id": [f"ex_{i:05d}" for i in range(1, n + 1)],
    "split": rng.choice(
        ["train", "validation", "test", "shifted_deployment"],
        size=n,
        p=[0.45, 0.20, 0.20, 0.15],
    ),
})

base_difficulty = rng.beta(2.0, 5.0, size=n)

shift_penalty = np.where(
    evaluation["split"] == "shifted_deployment",
    0.18,
    0.0,
)

evaluation["true_label"] = rng.binomial(1, 0.45, size=n)

signal_strength = np.where(
    evaluation["true_label"] == 1,
    0.70,
    0.30,
)

noise = rng.normal(0, 0.14 + shift_penalty, size=n)

evaluation["predicted_probability"] = np.clip(
    signal_strength - 0.20 * base_difficulty + noise,
    0.01,
    0.99,
)

evaluation["prediction"] = (evaluation["predicted_probability"] >= 0.50).astype(int)
evaluation["correct"] = (evaluation["prediction"] == evaluation["true_label"]).astype(int)

evaluation.head()

In [ ]:
summary = (
    evaluation
    .groupby("split")
    .agg(
        examples=("example_id", "count"),
        accuracy=("correct", "mean"),
        mean_confidence=("predicted_probability", "mean"),
    )
    .reset_index()
)

train_accuracy = summary.loc[summary["split"] == "train", "accuracy"].iloc[0]
test_accuracy = summary.loc[summary["split"] == "test", "accuracy"].iloc[0]
shift_accuracy = summary.loc[summary["split"] == "shifted_deployment", "accuracy"].iloc[0]

diagnostics = pd.DataFrame([
    {"metric": "train_minus_test_gap", "value": train_accuracy - test_accuracy},
    {"metric": "test_minus_shifted_deployment_gap", "value": test_accuracy - shift_accuracy},
])

summary, diagnostics

## Interpretation

Validation asks whether apparent performance survives outside the training sample and under shifted deployment conditions.